# Construct close-election RD sample (vote-margin)
This notebook defines market-oriented blocs, constructs vote-share running
variables, and links post-election cabinets for the close-election design.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.elections_parlgov import (
    compute_bloc_ideology_distance,
    compute_cabinet_ideology,
    compute_market_seat_shares,
    compute_market_vote_shares,
    compute_top2_margin_by_bloc,
    select_post_election_cabinets,
)
from src.paths import ANALYSIS_DIR, CLEAN_DIR, INTERMEDIATE_DIR, PAPER_LOGS_DIR
from src.qc import assert_unique_key
from src.viz_style import set_style

In [2]:
set_style()

panel_path = CLEAN_DIR / "panel_annual_atlas.parquet"
if not panel_path.exists():
    raise FileNotFoundError("Missing annual panel. Run 03b_build_annual_panel first.")

panel = pd.read_parquet(panel_path)


elections_path = INTERMEDIATE_DIR / "parlgov_elections.parquet"
results_path = INTERMEDIATE_DIR / "parlgov_election_results.parquet"
cabinets_path = INTERMEDIATE_DIR / "parlgov_cabinets.parquet"
cabinet_parties_path = INTERMEDIATE_DIR / "parlgov_cabinet_parties.parquet"
parties_path = INTERMEDIATE_DIR / "parlgov_parties.parquet"

for path in [
    elections_path,
    results_path,
    cabinets_path,
    cabinet_parties_path,
    parties_path,
]:
    if not path.exists():
        raise FileNotFoundError(f"Missing ParlGov output: {path}")

elections = pd.read_parquet(elections_path)
results = pd.read_parquet(results_path)
cabinets = pd.read_parquet(cabinets_path)
cabinet_parties = pd.read_parquet(cabinet_parties_path)
parties = pd.read_parquet(parties_path)

In [3]:
MARKET_THRESHOLD = 5.0
ALT_THRESHOLD = 6.0
START_YEAR = 2000

max_year = int(panel["year"].max()) - 3

elections = elections[elections["election_type"].str.contains("Parliament", case=False, na=False)].copy()
elections = elections[elections["year"].notna()].copy()
elections["year"] = elections["year"].astype(int)
elections = elections[(elections["year"] >= START_YEAR) & (elections["year"] <= max_year)].copy()

In [4]:
results = results.dropna(subset=["seats"]).copy()

seat_share_main = compute_market_seat_shares(results, parties, threshold=MARKET_THRESHOLD)
seat_share_alt = compute_market_seat_shares(results, parties, threshold=ALT_THRESHOLD).rename(
    columns={
        "seat_market": "seat_market_alt",
        "seat_share_market": "seat_share_market_alt",
    }
)

vote_share_main = compute_market_vote_shares(results, parties, threshold=MARKET_THRESHOLD)
vote_share_alt = compute_market_vote_shares(results, parties, threshold=ALT_THRESHOLD).rename(
    columns={
        "vote_share_market_raw": "vote_share_market_raw_alt",
        "vote_share_market": "vote_share_market_alt",
    }
)

ideology_distance = compute_bloc_ideology_distance(results, parties, threshold=MARKET_THRESHOLD)
top2_margin = compute_top2_margin_by_bloc(results, parties, threshold=MARKET_THRESHOLD)

In [5]:
elections = elections.merge(seat_share_main, on="election_id", how="left")
elections = elections.merge(
    seat_share_alt[["election_id", "seat_market_alt", "seat_share_market_alt"]],
    on="election_id",
    how="left",
)
elections = elections.merge(vote_share_main, on="election_id", how="left")
elections = elections.merge(
    vote_share_alt[["election_id", "vote_share_market_alt"]],
    on="election_id",
    how="left",
)
elections = elections.merge(
    ideology_distance[["election_id", "lr_market", "lr_nonmarket", "lr_distance", "lr_distance_abs"]],
    on="election_id",
    how="left",
)
elections = elections.merge(
    top2_margin[
        [
            "election_id",
            "top_market_share",
            "top_nonmarket_share",
            "top2_margin",
            "top2_margin_abs",
            "winner_market_top2",
        ]
    ],
    on="election_id",
    how="left",
)

# Running variables

elections["running_var_seat"] = elections["seat_share_market"] - 0.5
elections["running_var_seat_alt"] = elections["seat_share_market_alt"] - 0.5
elections["running_var_vote"] = elections["vote_share_market"] - 0.5
elections["running_var_vote_margin"] = 2 * elections["vote_share_market"] - 1
elections["running_var_vote_alt"] = elections["vote_share_market_alt"] - 0.5
elections["running_var_top2"] = elections["top2_margin"]

# Treatment indicators

elections["market_majority_seat"] = (elections["running_var_seat"] > 0).astype(int)
elections["market_majority_vote"] = (elections["running_var_vote"] > 0).astype(int)
elections["market_majority_vote_alt"] = (elections["running_var_vote_alt"] > 0).astype(int)
elections["market_majority_top2"] = (elections["running_var_top2"] > 0).astype(int)

In [6]:
# Cabinet ideology and incumbency
cabinet_summary = compute_cabinet_ideology(
    cabinets,
    cabinet_parties,
    parties,
    results,
    threshold=MARKET_THRESHOLD,
)

post_cabinets = select_post_election_cabinets(cabinets)
post_cabinets = post_cabinets.merge(
    cabinet_summary,
    left_on="post_cabinet_id",
    right_on="cabinet_id",
    how="left",
)
post_cabinets = post_cabinets.rename(
    columns={
        "cabinet_lr": "post_cabinet_lr",
        "cabinet_market_share": "post_cabinet_market_share",
        "cabinet_party_count": "post_cabinet_party_count",
    }
)
post_cabinets["winner_market"] = (post_cabinets["post_cabinet_lr"] >= MARKET_THRESHOLD).astype(float)

incumbent = post_cabinets.merge(
    cabinet_summary,
    left_on="previous_cabinet_id",
    right_on="cabinet_id",
    how="left",
    suffixes=("", "_incumbent"),
)
incumbent = incumbent.rename(
    columns={
        "cabinet_lr": "incumbent_cabinet_lr",
        "cabinet_market_share": "incumbent_cabinet_market_share",
        "cabinet_party_count": "incumbent_cabinet_party_count",
    }
)
incumbent["incumbent_market"] = (incumbent["incumbent_cabinet_lr"] >= MARKET_THRESHOLD).astype(float)

cabinet_cols = [
    "election_id",
    "post_cabinet_id",
    "previous_cabinet_id",
    "post_cabinet_lr",
    "post_cabinet_market_share",
    "post_cabinet_party_count",
    "winner_market",
    "incumbent_cabinet_lr",
    "incumbent_cabinet_market_share",
    "incumbent_cabinet_party_count",
    "incumbent_market",
]

elections = elections.merge(incumbent[cabinet_cols], on="election_id", how="left")
elections["market_switch"] = (elections["winner_market"] != elections["incumbent_market"]).astype(float)

In [7]:
# Keep one parliamentary election per country-year

elections = elections.sort_values("election_date")
elections = elections.groupby(["iso3c", "year"], as_index=False).tail(1).reset_index(drop=True)

In [8]:
events = elections.rename(columns={"year": "election_year"})
assert_unique_key(events, ["iso3c", "election_year"])

In [9]:
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
sample_path = ANALYSIS_DIR / "close_elections_vote_margin.parquet"
legacy_path = ANALYSIS_DIR / "close_elections_sample.parquet"

events.to_parquet(sample_path, index=False)
events.to_parquet(legacy_path, index=False)

coverage = pd.DataFrame(
    {
        "rows": [len(events)],
        "countries": [events["iso3c"].nunique()],
        "min_year": [events["election_year"].min()],
        "max_year": [events["election_year"].max()],
        "share_market_majority_vote": [events["market_majority_vote"].mean()],
        "share_market_majority_seat": [events["market_majority_seat"].mean()],
    }
)
display(coverage.style.set_caption("Close-election sample summary"))
display(events.head(5).style.set_caption("Sample rows"))

running = events["running_var_vote"].dropna()
heaping_share = (running.round(2) == running).mean() if not running.empty else np.nan
missing_vote_share = events["vote_share_market"].isna().mean()
missing_top2 = events["running_var_top2"].isna().mean()

meta = {
    "build": {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "pipeline_version": "rd-iv-vote-margin-v1",
    },
    "definitions": {
        "market_threshold": MARKET_THRESHOLD,
        "alt_threshold": ALT_THRESHOLD,
        "running_variable": "vote_share_market - 0.5",
    },
    "inputs": {
        "panel_annual": str(panel_path),
        "parlgov_elections": str(elections_path),
        "parlgov_results": str(results_path),
        "parlgov_cabinets": str(cabinets_path),
        "parlgov_parties": str(parties_path),
    },
    "summary": {
        **coverage.to_dict(orient="records")[0],
        "missing_vote_share": missing_vote_share,
        "missing_top2_margin": missing_top2,
    },
    "qc": {"running_var_vote_heaping_share": heaping_share},
}

meta_path = PAPER_LOGS_DIR / "close_elections_vote_margin_metadata.json"
meta_path.parent.mkdir(parents=True, exist_ok=True)
meta_path.write_text(json.dumps(meta, indent=2))

,rows,countries,min_year,max_year,share_market_majority_vote,share_market_majority_seat
0,305,37,2000,2021,0.662295,0.675410


,election_id,type_id,country_id,date,first_round_election_id,early,wikipedia,seats_total,electorate,votes_cast,votes_valid,data_source,description,comment,previous_parliament_election_id,previous_ep_election_id,previous_cabinet_id_x,old_countryID,old_parlID,country_name,iso3c,iso_numeric,election_date,election_year,election_type,seat_total,seat_market,seat_share_market,seat_market_alt,seat_share_market_alt,vote_share_total,vote_share_market_raw,vote_share_market,vote_share_market_alt,lr_market,lr_nonmarket,lr_distance,lr_distance_abs,top_market_share,top_nonmarket_share,top2_margin,top2_margin_abs,winner_market_top2,running_var_seat,running_var_seat_alt,running_var_vote,running_var_vote_margin,running_var_vote_alt,running_var_top2,market_majority_seat,market_majority_vote,market_majority_vote_alt,market_majority_top2,post_cabinet_id,previous_cabinet_id_y,post_cabinet_lr,post_cabinet_market_share,post_cabinet_party_count,winner_market,incumbent_cabinet_lr,incumbent_cabinet_market_share,incumbent_cabinet_party_count,incumbent_market,market_switch
0,797,13,62,2000-01-03,nan,0,"http://en.wikipedia.org/wiki/Croatian_parliamentary_election,_2000",151,3686378.000000,2821020.000000,2774275.000000,no-2010,None,None,nan,nan,nan,nan,nan,Croatia,HRV,191,2000-01-03 00:00:00,2000,Parliamentary election,151.000000,92.000000,0.609272,92.000000,0.609272,92.600000,49.200000,0.531317,0.516199,7.001601,3.245600,3.756001,3.756001,0.244000,0.408000,-0.164000,0.164000,0.000000,0.109272,0.109272,0.031317,0.062635,0.016199,-0.164000,1,1,1,0,1049.000000,1049.000000,4.777405,0.440860,6.000000,0.000000,4.777405,0.440860,6.000000,0.000000,0.000000
1,687,13,27,2000-03-12,nan,0,"http://en.wikipedia.org/wiki/Spanish_general_election,_2000",350,33969640.000000,23339490.000000,22814467.000000,epp,None,None,99.000000,585.000000,370.000000,724.000000,20000.000000,Spain,ESP,724,2000-03-12 00:00:00,2000,Parliamentary election,350.000000,209.000000,0.597143,205.000000,0.585714,97.080000,52.130000,0.536980,0.525752,7.412676,3.493823,3.918853,3.918853,0.452400,0.347100,0.105300,0.105300,1.000000,0.097143,0.085714,0.036980,0.073960,0.025752,0.105300,1,1,1,1,309.000000,370.000000,7.596900,1.000000,1.000000,1.000000,7.596900,1.000000,1.000000,1.000000,0.000000
2,475,13,41,2000-04-09,nan,0,"http://en.wikipedia.org/wiki/Greek_legislative_election,_2000",300,9372541.000000,7026527.000000,6868011.000000,mig,None,None,399.000000,482.000000,33.000000,300.000000,20000.000000,Greece,GRC,300,2000-04-09 00:00:00,2000,Parliamentary election,300.000000,125.000000,0.416667,125.000000,0.416667,97.940000,42.740000,0.436390,0.436390,6.736500,3.965163,2.771337,2.771337,0.427400,0.437900,-0.010500,0.010500,0.000000,-0.083333,-0.083333,-0.063610,-0.127221,-0.063610,-0.010500,0,0,0,0,190.000000,33.000000,4.496800,0.000000,1.000000,0.000000,4.496800,0.000000,1.000000,0.000000,0.000000
3,237,13,5,2000-06-25,nan,1,"http://en.wikipedia.org/wiki/Japanese_general_election,_2000",480,100492328.000000,62757828.000000,59844601.000000,parline,None,None,628.000000,nan,211.000000,392.000000,20000.000000,Japan,JPN,392,2000-06-25 00:00:00,2000,Parliamentary election,480.000000,421.000000,0.877083,263.000000,0.547917,99.820000,78.980000,0.791224,0.409036,6.744925,1.823702,4.921223,4.921223,0.283100,0.112300,0.170800,0.170800,1.000000,0.377083,0.047917,0.291224,0.582448,-0.090964,0.170800,1,1,0,1,163.000000,211.000000,7.764001,1.000000,3.000000,1.000000,8.016504,1.000000,3.000000,1.000000,0.000000
4,59,13,15,2000-10-08,nan,0,"http://en.wikipedia.org/wiki/Lithuanian_parliamentary_election,_2000",141,2626321.000000,1539743.000000,1471247.000000,vrk,None,2000 election with plurality instead of majority run-off for majority tier of parallel system,566.000000,nan,735.000000,440.000000,20000.000000,Lithuania,LTU,440,2000-10-08 00:00:00,2000,Parliamentary election,141.000000,49.000000,0.347518,49.000000,0.347518,98.640000,37.560000,0.380779,0.380779,7.371472,3.686107,3.685364,3.685364,0.172500,0.310800,-0.138300

1229

## Interpretation
The close-election sample now uses vote shares to construct a continuous
running variable, while preserving cabinet ideology and incumbency markers
required for positive vs negative shock splits.